# 🏠🏠🏠 Projet Kaggle : Catboost : Selection de variable 🏠🏠🏠

## Initialisation

### Importation des bibliothèques nécessaires


In [1]:
import json
import re

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, EFeaturesSelectionAlgorithm, EShapCalcType, Pool
from sklearn.model_selection import train_test_split

### Importation des données


In [2]:
with open("../data/processed/dtype_dict.json") as f:
    dtype_dict = json.load(f)

train = pd.read_csv(
    "../data/processed/train.csv",
    delimiter=",",
    encoding="utf-8",
    index_col="Id",
    dtype=dtype_dict,
)

test = pd.read_csv(
    "../data/processed/test.csv",
    delimiter=",",
    encoding="utf-8",
    index_col="Id",
    dtype=dtype_dict,
)

dfs = [train, test]

### Reprise des transformations intéressantes


In [3]:
neighborhoods_to_keep = [
    "Brookside",
    "Clear Creek",
    "Crawford",
    "Northridge",
    "Northridge Heights",
    "Stone Brook",
    "Veenker",
]

for df in dfs:
    df["Neighborhood_agg2"] = np.where(
        df["Neighborhood"].isin(neighborhoods_to_keep), df["Neighborhood"], "Autre"
    )

    df["FullBath_tot"] = df["FullBath"] + df["BsmtFullBath"]
    df["HalfBath_tot"] = df["HalfBath"] + df["BsmtHalfBath"]

    # Rajout d'un traitement pour Alley
    df["Alley"] = df["Alley"].fillna("No Alley")

    # Création d'un vecteur pour representer les mois
    df["MoSold_enc"] = list(zip(df["SinMoSold"], df["CosMoSold"]))

## Sélection de Caractéristiques

### Définition des Combinaisons de Variables

L'objectif est de sélectionner des variables non corrélées qui apportent des informations complémentaires. Par exemple, les agrégations et les encodages ordinaux doivent être testés indépendamment. Le même principe s'applique aux corrélations par construction (par exemple, GrLivArea = 1stFlrSF + 2ndFlrSF). Pour débuter la Récursive Feature Elimination (RFE), je souhaite effectuer une sélection initiale stratégique basée sur mes premières observations.

### La Désillusion

Cependant, la combinatoire des variables est trop importante pour être exhaustive. Je vais donc choisir une base de variables et me concentrer sur deux approches : une RFE sans agrégation et une autre avec agrégation. Cette décision sera guidée par une analyse des valeurs de Shapley pour affiner l'interprétation. Mon intuition est de privilégier des découpages qui ont du sens tout en conservant une granularité fine dans l'information et de regrouper dans un deuxième temps.

### Variables à supprimer


In [4]:
# Salle de bain et toilettes
# On garde uniquement FullBath_tot et HalfBath_tot
col_suppr = [
    "HalfBath",
    "HalfBath_optb",
    "FullBath",
    "FullBath_optb",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtFullBath_optb",
    "BsmtHalfBath_optb",
]

# Les surfaces je vais garder GarageCars (ce découpage semble vraiment pertinent et remonte dans les features importances)
# j'enleve dans un premier temps 2ndFlrSF
col_suppr.extend(["GarageArea", "2ndFlrSF"])


# Le style de maisons
# je vais garder uniquement BldgType car les dates et le fait d'avoir un étage ou non semble déjà indiqué par d'autrs variables.
# Les configuirations atypique de peuvent de cette manière resortir
col_suppr.extend(["MSSubClass", "HouseStyle"])

# Tout ce qui fini par "_agg", "_agg2", "_ord" ou encore "_optb" (soit ces agregation seront reutilisées dans un deuxième temps, soit elle seront chalengées par un decoupage basé sur les valeurs de Shapley)
for col in train.columns:
    if re.search(r"_(ord|agg(|2)|optb|tot|)$", col):
        col_suppr.append(col)

# les mois, je garde uniquement un embedding et le label (ne sera pas pris en compte dans la modélisation)
col_suppr.extend(["SinMoSold", "CosMoSold"])

train_1 = train.drop(columns=col_suppr, axis=1).copy()

### Colonnes et index des variables categorielles


In [5]:
# Colonnes de type catégoriel
categorical_columns = (
    train_1.drop(columns=["SalePrice", "MoSold", "MoSold_enc"], axis=1)
    .select_dtypes(include=["category", "object"])
    .columns
).to_list()

### Séparation en train test


In [6]:
df_train, df_test = train_test_split(train_1, test_size=0.25, random_state=42)

### Création des Pools


In [7]:
# Création des objets Pool pour CatBoost
train_pool = Pool(
    df_train.drop(columns=["SalePrice", "MoSold"], axis=1),
    label=df_train["SalePrice"],
    embedding_features=["MoSold_enc"],
    cat_features=categorical_columns,
)
test_pool = Pool(
    df_test.drop(columns=["SalePrice", "MoSold"], axis=1),
    label=df_test["SalePrice"],
    embedding_features=["MoSold_enc"],
    cat_features=categorical_columns,
)

### Modèle Catboost


In [8]:
# Initialiser et entraîner le modèle CatBoostClassifier
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.035,
    depth=8,
    loss_function="RMSE",
    verbose=100,
    use_best_model=True,
)

### Entrainement du modèle


In [9]:
summary = model.select_features(
    train_pool,
    eval_set=test_pool,
    features_for_select="0-70",
    num_features_to_select=10,
    steps=3,
    algorithm=EFeaturesSelectionAlgorithm.RecursiveByShapValues,
    shap_calc_type=EShapCalcType.Regular,
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

### Remarques et Ajustements

Bien que cela puisse légèrement affecter les performances, je choisirais de retirer les dix premières variables pour plusieurs raisons :

- BsmtUnfSF : Corrélé avec d'autres variables de surface, spécifiquement pour le sous-sol.
- 3SsnPorch : Trop déséquilibré.
- RoofMatl : Idem.
- CentralAir : Idem.
- PoolArea : PoolQC devrait suffire.
- LandSlope : Trop déséquilibré et corrélé avec LotShape.
- KitchenAbvGr : L'interprétation avec les valeurs de Shapley est contre-intuitive dans le premier modèle. L'interprétation avec la qualité de la cuisine est plus claire.
- Utilities : Trop déséquilibré.
- MiscVal : Corrélé avec MiscFeature, qui devrait suffire.
- LowQualFinSF : Corrélé avec d'autres variables de surface, spécifiquement pour le sous-sol.
  En termes de performance, si l'on se base sur la RMSE, il serait préférable de s'arrêter à LandSlope.

## Test avec des Agrégations

Deuxième RFE avec aggregations de modalités en fonction des valeurs de Shapley et des précédents tests
Reprise de certains regroupements existants et création de variables


In [10]:
dfs = [train, test]

for df in dfs:
    df["BsmtQual_agg"] = df["BsmtQual"].replace(
        {
            "Fair (70-79 inches)": "Fair/Typical (70-89 inches)",
            "Typical (80-89 inches)": "Fair/Typical (70-89 inches)",
        }
    )

    df["Fireplaces_agg"] = df["Fireplaces"].replace(
        {0: "0", 1: "1", 2: "2 et plus", 3: "2 et plus"}
    )

    # Le fait de ne pas avoir de sous-sol signifie qu'il n'y a pas d'exposition, par définition
    # Ce regroupement semble logique mais je n'y ai pas reflechi au début du projet
    df["BsmtExposure_agg"] = df["BsmtExposure"].replace("No basement", "No Exposure")

    df["BsmtCond_agg"] = df["BsmtCond"].replace(
        {
            "Fair - dampness or some cracking or settling": "Fair/Poor",
            "Poor - Severe cracking, settling, or wetness": "Fair/Poor",
        }
    )

    df["GarageCond_agg"] = df["GarageCond"].replace(
        {
            "Fair": "Fair/Poor",
            "Poor": "Fair/Poor",
            "Good": "Good/Excellent",
            "Excellent": "Good/Excellent",
        }
    )

In [11]:
# Salle de bain et toilettes
# On garde uniquement FullBath_tot et HalfBath_tot
col_suppr_2 = [
    "HalfBath",
    "HalfBath_optb",
    "FullBath",
    "FullBath_optb",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtFullBath_optb",
    "BsmtHalfBath_optb",
]

# Les surfaces je vais garder GarageCars (ce découpage semble vraiment pertinent et remonte dans les features importances)
# j'enleve dans un premier temps 2ndFlrSF
col_suppr_2.extend(["GarageArea", "2ndFlrSF"])


# Le style de maisons
# je vais garder uniquement BldgType car les dates et le fait d'avoir un étage ou non semble déjà indiqué par d'autrs variables.
# Les configuirations atypique de peuvent de cette manière resortir
col_suppr_2.extend(["MSSubClass", "HouseStyle"])

# Tout ce qui fini par "_agg2", "_ord" ou encore "_optb" (les agregations non utilisées)
for col in train.columns:
    if re.search(r"_(ord|agg2|optb|tot)$", col):
        col_suppr_2.append(col)

# Selection des agregations à la place
col_suppr_2.extend(
    [
        "OverallQual",
        "OverallCond",
        "HeatingQC",
        "FireplaceQu",
        "LotShape",
        "Condition1",
        "Functional",
        "GarageQual",
        "Exterior1st",
        "Exterior2nd",
        "LotConfig",
        "GarageCond",
        "BsmtQual",
        "Fireplaces",
        "BsmtExposure",
        "BsmtCond",
    ]
)

# les mois, je garde uniquement un embedding et le label (ne sera pas pris en compte dans la modélisation)
col_suppr_2.extend(["SinMoSold", "CosMoSold"])

train_2 = train.drop(columns=col_suppr_2, axis=1).copy()

In [12]:
# Colonnes de type catégoriel
categorical_columns = (
    train_2.drop(columns=["SalePrice", "MoSold", "MoSold_enc"], axis=1)
    .select_dtypes(include=["category", "object"])
    .columns
).to_list()

### Séparation en train test


In [13]:
df_train_2, df_test_2 = train_test_split(train_2, test_size=0.25, random_state=42)

### Création des Pools


In [14]:
# Création des objets Pool pour CatBoost
train_pool_2 = Pool(
    df_train_2.drop(columns=["SalePrice", "MoSold"], axis=1),
    label=df_train_2["SalePrice"],
    embedding_features=["MoSold_enc"],
    cat_features=categorical_columns,
)
test_pool_2 = Pool(
    df_test_2.drop(columns=["SalePrice", "MoSold"], axis=1),
    label=df_test_2["SalePrice"],
    embedding_features=["MoSold_enc"],
    cat_features=categorical_columns,
)

### Modèle Catboost


In [15]:
# Initialiser et entraîner le modèle CatBoostClassifier
model2 = CatBoostRegressor(
    iterations=500,
    learning_rate=0.035,
    depth=8,
    loss_function="RMSE",
    verbose=100,
    use_best_model=True,
)

### Entrainement du modèle


In [16]:
summary2 = model2.select_features(
    train_pool_2,
    eval_set=test_pool_2,
    features_for_select="0-71",
    num_features_to_select=10,
    steps=3,
    algorithm=EFeaturesSelectionAlgorithm.RecursiveByShapValues,
    shap_calc_type=EShapCalcType.Regular,
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

### Remarques et Ajustements

Le modèle semble plus performant avec des variables agrégées. Dans ce cas, la sélection peut se faire à la performance. Voici les variables que l'on peut enlever :

- EnclosedPorch
- 3SsnPorch
- Utilities
- CentralAir
- Fence
- Street
- PoolArea
- Foundation
- MiscVal
- LotConfig_agg
- BsmtUnfSF
- KitchenAbvGr
- LowQualFinSF
- LandSlope
- MasVnrType
- Exterior2nd_agg
- MSZoning
- HeatingQC_agg
- Heating
- DiffYearsGarageBuilt
- RoofMatl
- Electrical
- MiscFeature
- DiffYearsRemodAddBuilt
- BsmtFinSF2
- BsmtFinType2
- Neighborhood
- ScreenPorch
- PoolQC
- WoodDeckSF
- SaleType
- RoofStyle
- LotFrontage
- BedroomAbvGr
- FireplaceQu_agg
- Alley
- YrSold
- LandContour
- Condition2

## Test avec des encodages ordinaux

Troisième RFE avec agrégations de modalités et encodages ordinaux lorsque cela est possible.


In [17]:
col_suppr_3 = col_suppr_2.copy()

for col in train.columns:
    if col.endswith("_ord"):
        col_rep = col.removesuffix("_ord")
        col_suppr_3.remove(col)
        for col2 in train.columns:
            if col2.startswith(col_rep) and col2 != col:
                col_suppr_3.append(col2)

train_3 = train.drop(columns=col_suppr_3, axis=1).copy()

In [18]:
# Colonnes de type catégoriel
categorical_columns = (
    train_3.drop(columns=["SalePrice", "MoSold", "MoSold_enc"], axis=1)
    .select_dtypes(include=["category", "object"])
    .columns
).to_list()

### Séparation en train test


In [19]:
df_train_3, df_test_3 = train_test_split(train_3, test_size=0.25, random_state=42)

### Création des Pools


In [20]:
# Création des objets Pool pour CatBoost
train_pool_3 = Pool(
    df_train_3.drop(columns=["SalePrice", "MoSold"], axis=1),
    label=df_train_3["SalePrice"],
    embedding_features=["MoSold_enc"],
    cat_features=categorical_columns,
)

test_pool_3 = Pool(
    df_test_3.drop(columns=["SalePrice", "MoSold"], axis=1),
    label=df_test_3["SalePrice"],
    embedding_features=["MoSold_enc"],
    cat_features=categorical_columns,
)

### Modèle Catboost


In [21]:
# Initialiser et entraîner le modèle CatBoostClassifier
model3 = CatBoostRegressor(
    iterations=500,
    learning_rate=0.035,
    depth=8,
    loss_function="RMSE",
    verbose=100,
    use_best_model=True,
)

### Entrainement du modèle


In [22]:
summary3 = model3.select_features(
    train_pool_3,
    eval_set=test_pool_3,
    features_for_select="0-71",
    num_features_to_select=10,
    steps=3,
    algorithm=EFeaturesSelectionAlgorithm.RecursiveByShapValues,
    shap_calc_type=EShapCalcType.Regular,
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

### Remarques et Ajustements

Le modèle semble plus performant avec des variables agrégées, en particulier avec les encodage ordinaux. Dans ce cas, la sélection peut se faire à la performance. Voici les variables que l'on peut enlever :

- GarageCond_ord
- MSZoning
- Exterior2nd_agg  
- BsmtUnfSF  
- LandContour_ord  
- PoolArea  
- 3SsnPorch  
- EnclosedPorch  
- MasVnrType  
- LandSlope_ord  
- KitchenAbvGr  
- LowQualFinSF  
- Utilities_ord  
- BsmtFinType2_ord  
- MiscVal  
- RoofMatl  
- CentralAir  
- Street  
- YrSold  
- DiffYearsGarageBuilt  
- LotConfig_agg  
- BsmtFinSF2  
- Neighborhood  
- Fence  
- LotShape_ord  
- DiffYearsRemodAddBuilt  
- Heating  
- BedroomAbvGr  
- Functional_ord  
- ExterQual_ord  
- RoofStyle  
- MiscFeature  
- ScreenPorch  
- GarageQual_ord  
- MasVnrArea  
- Alley
- BsmtQual_ord
- Electrical
- Condition2

De manière à obtenir une certaine confiance en supprimant la variable, nous pourrions essayer de voir quelles variables sont supprimées dans chacune des listes ci-dessus

In [47]:
def find_common_removed_features(dicts:list, x:int):
    # Initialize the common removed features storage
    common_removed_features = set(dicts[0]['eliminated_features_names'][:x])

    # Initialize a set to store common removed features
    for dict in dicts:
        # Extract the top x removed features from each summary
        removed_features = set(dict['eliminated_features_names'][:x])
        # Find common features removed in all three summaries
        common_removed_features = common_removed_features & removed_features
    
    # Print the common features
    if common_removed_features:
        print(f"{len(common_removed_features)} Common features removed in the top", x, "of each summary:")
        for feature in common_removed_features:
            print(feature)
    else:
        print("No common features removed in the top", x, "of each summary.")

In [51]:
find_common_removed_features([summary2, summary3], 35)

24 Common features removed in the top 35 of each summary:
LotConfig_agg
Fence
EnclosedPorch
MiscFeature
MiscVal
MSZoning
DiffYearsRemodAddBuilt
Neighborhood
CentralAir
PoolArea
RoofStyle
3SsnPorch
BsmtFinSF2
Exterior2nd_agg
RoofMatl
ScreenPorch
Street
DiffYearsGarageBuilt
BedroomAbvGr
MasVnrType
KitchenAbvGr
Heating
BsmtUnfSF
LowQualFinSF
